In [1]:
import sys
sys.path.insert(0, '/app')
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [2]:
    

import numpy as np
import pandas as pd
from pathlib import Path

from core.data_org import BUNDLE_DIR
from strat.strat_backtest import (
    ETF_LIST,
    compute_dual_momentum,
    get_current_allocation,
)
from strat.strat_analysis import (
    print_allocation_summary,
    print_metrics,
    compute_period_returns,
)
from strat.strat_visualise import (
    ETF_COLORS,
    plot_normalized_prices,
    create_allocation_gif,
    plot_pnl_histogram,
    plot_portfolio_value,
    plot_allocation_history,
    plot_cumulative_pnl,
    plot_monthly_returns_chart,
)
from backtest.dual_momentum_backtest import (
    generate_orders_from_allocations,
    compute_strategy_performance,
    save_backtest_diagnostics,
)

INPUT_FILE = Path("/data/bundle/test_etf_features_bundle.parquet")
OUTPUT_DIR = Path("/data/work/dual_momentum/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input file: {INPUT_FILE}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"ETF List: {ETF_LIST}")

Input file: /data/bundle/test_etf_features_bundle.parquet
Output directory: /data/work/dual_momentum
ETF List: ['QQQ', 'SPY', 'TLT', 'GLD', 'VWO']


## Configuration

Configure strategy and backtest parameters:

In [3]:
STRATEGY_PARAMS = {
    'lookback': 460,  # ROC lookback period in bars
    'default_etf_idx': 2,  # TLT (safe haven)
    'top_n': 2,
    'abs_momentum_threshold': 0.01,
}

# Construct feature_id from lookback
STRATEGY_PARAMS['feature_id'] = f"F_roc_{STRATEGY_PARAMS['lookback']}_F_mid_f32_f16"

BACKTEST_PARAMS = {
    'rebalance_threshold': 0.05,
}

print("Strategy Parameters:")
for k, v in STRATEGY_PARAMS.items():
    print(f"  {k}: {v}")
print("\nBacktest Parameters:")
for k, v in BACKTEST_PARAMS.items():
    print(f"  {k}: {v}")

Strategy Parameters:
  lookback: 460
  default_etf_idx: 2
  top_n: 2
  abs_momentum_threshold: 0.01
  feature_id: F_roc_460_F_mid_f32_f16

Backtest Parameters:
  rebalance_threshold: 0.05


## Step 1: Load Data

Load the ETF features bundle:

In [4]:
df = pd.read_parquet(INPUT_FILE)
#df = df[48000:]
print(f"Loaded: {df.shape}")
print(f"Index: {df.index.name}, dtype: {df.index.dtype}")
print(f"Date range: {df.index.min()} to {df.index.max()}")

etfs_in_bundle = sorted(set(c.split("_")[0] for c in df.columns if "_F_" in c))
print(f"\nETFs in bundle: {etfs_in_bundle}")

# Show available lookback values (from F_roc_* columns)
roc_cols = [c for c in df.columns if "_roc_" in c]
available_lookbacks = sorted(set(int(c.split("_")[3]) for c in roc_cols))
print(f"Available ROC lookbacks (bars): {available_lookbacks}")

feature_cols = [c for c in df.columns if STRATEGY_PARAMS['feature_id'] in c]
print(f"Feature columns found: {len(feature_cols)}")

Loaded: (48107, 611)
Index: None, dtype: int64
Date range: 0 to 48106

ETFs in bundle: ['GLD', 'QQQ', 'SPY', 'TLT', 'VWO']
Available ROC lookbacks (bars): [14, 60, 240]
Feature columns found: 0


In [27]:
!pip install numba


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
# Generate ROC for additional lookback periods
from features.feature_ta_utils import numba_roc

# Define periods: 1-7, 10, 20, 30, ..., 2400
periods = [1, 2, 3, 4, 5, 6, 7, 10] + list(range(20, 2410, 10))
print(f"Generating ROC for {len(periods)} periods: {periods[:10]}...{periods[-3:]}")

etfs = ['QQQ', 'SPY', 'TLT', 'GLD', 'VWO']
mid_col = "F_mid_f32"

# Build all new columns first, then concat once to avoid fragmentation
new_cols = {}
for etf in etfs:
    mid_price_col = f"{etf}_{mid_col}"
    
    for period in periods:
        col_name = f"{etf}_F_roc_{period}_{mid_col}_f16"
        if col_name not in df.columns:
            new_cols[col_name] = numba_roc(df[mid_price_col].to_numpy(), period)

# Add all columns at once
df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)

print(f"DataFrame shape after ROC generation: {df.shape}")

# Update available lookbacks
roc_cols = [c for c in df.columns if "_roc_" in c]
available_lookbacks = sorted(set(int(c.split("_")[3]) for c in roc_cols))
print(f"Available ROC lookbacks: {len(available_lookbacks)} periods")
print(f"  First 10: {available_lookbacks[:10]}")
print(f"  Last 10: {available_lookbacks[-10:]}")

Generating ROC for 247 periods: [1, 2, 3, 4, 5, 6, 7, 10, 20, 30]...[2380, 2390, 2400]
DataFrame shape after ROC generation: (48107, 1836)
Available ROC lookbacks: 248 periods
  First 10: [1, 2, 3, 4, 5, 6, 7, 10, 14, 20]
  Last 10: [2310, 2320, 2330, 2340, 2350, 2360, 2370, 2380, 2390, 2400]


## Step 2: Run Dual Momentum Strategy

Compute allocations using the dual momentum strategy:

In [6]:
df_allocations = compute_dual_momentum(
    p_df=df,
    p_feature_id=STRATEGY_PARAMS['feature_id'],
    p_default_etf_idx=STRATEGY_PARAMS['default_etf_idx'],
    p_top_n=STRATEGY_PARAMS['top_n'],
    p_abs_momentum_threshold=STRATEGY_PARAMS['abs_momentum_threshold'],
)

print(f"Allocations computed: {df_allocations.shape}")
print(f"\nAllocation columns: {[c for c in df_allocations.columns if c.startswith('A_')]}")

Allocations computed: (48107, 1848)

Allocation columns: ['A_QQQ_alloc', 'A_SPY_alloc', 'A_TLT_alloc', 'A_GLD_alloc', 'A_VWO_alloc', 'A_top_etf', 'A_n_positive_momentum', 'A_QQQ_roc_rank', 'A_SPY_roc_rank', 'A_TLT_roc_rank', 'A_GLD_roc_rank', 'A_VWO_roc_rank']


## Step 3: ROC Period Sweep

Loop over all ROC periods and compute performance for top 1 and top 2 assets.

In [ ]:
import matplotlib.pyplot as plt

def sweep_roc_periods(p_df, p_lookbacks, p_rebalance_threshold=0.05, p_abs_momentum_threshold=0.01, p_default_etf_idx=2):
    """
    Run dual momentum for different ROC periods and compute performance.
    
    Args:
        p_df: DataFrame with ROC columns and OHLC data
        p_lookbacks: List of ROC lookback periods to test
        p_rebalance_threshold: Rebalance threshold
        p_abs_momentum_threshold: Absolute momentum threshold
        p_default_etf_idx: Default ETF index (safe haven)
    
    Returns:
        dict with 'top1' and 'top2' lists of (period, total_return) tuples
    """
    from backtest.dual_momentum_backtest import generate_orders_from_allocations, compute_strategy_performance
    from strat.strat_backtest import compute_dual_momentum
    
    mid_col = "F_mid_f32"
    results = {
        'top1': [],
        'top2': [],
    }
    
    for lookback in p_lookbacks:
        feature_id = f"F_roc_{lookback}_{mid_col}_f16"
        
        # Compute allocations for this lookback
        df_alloc = compute_dual_momentum(
            p_df=p_df,
            p_feature_id=feature_id,
            p_default_etf_idx=p_default_etf_idx,
            p_top_n=1,
            p_abs_momentum_threshold=p_abs_momentum_threshold,
        )
        
        # Generate orders and compute performance for top_n=1
        orders_df, _ = generate_orders_from_allocations(df_alloc, p_rebalance_threshold)
        metrics_top1, _ = compute_strategy_performance(df_alloc, orders_df)
        results['top1'].append((lookback, metrics_top1['total_return']))
        
        # Compute allocations for top_n=2
        df_alloc2 = compute_dual_momentum(
            p_df=p_df,
            p_feature_id=feature_id,
            p_default_etf_idx=p_default_etf_idx,
            p_top_n=2,
            p_abs_momentum_threshold=p_abs_momentum_threshold,
        )
        
        orders_df2, _ = generate_orders_from_allocations(df_alloc2, p_rebalance_threshold)
        metrics_top2, _ = compute_strategy_performance(df_alloc2, orders_df2)
        results['top2'].append((lookback, metrics_top2['total_return']))
        
        print(f"Lookback {lookback:4d}: top1={metrics_top1['total_return']*100:7.2f}%, top2={metrics_top2['total_return']*100:7.2f}%")
    
    return results


# Get available lookbacks from the dataframe
roc_cols = [c for c in df.columns if "_roc_" in c]
available_lookbacks = sorted(set(int(c.split("_")[3]) for c in roc_cols))
print(f"Running ROC sweep over {len(available_lookbacks)} periods...")

# Run the sweep
results = sweep_roc_periods(
    p_df=df,
    p_lookbacks=available_lookbacks,
    p_rebalance_threshold=BACKTEST_PARAMS['rebalance_threshold'],
    p_abs_momentum_threshold=STRATEGY_PARAMS['abs_momentum_threshold'],
    p_default_etf_idx=STRATEGY_PARAMS['default_etf_idx'],
)

# Plot results
fig, ax = plt.subplots(figsize=(14, 6))

periods = [x[0] for x in results['top1']]
top1_returns = [x[1] * 100 for x in results['top1']]
top2_returns = [x[1] * 100 for x in results['top2']]

ax.plot(periods, top1_returns, 'r-', linewidth=2, label='Top 1 Asset', marker='o', markersize=3)
ax.plot(periods, top2_returns, 'b-', linewidth=2, label='Top 2 Assets', marker='s', markersize=3)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('ROC Lookback Period (bars)', fontsize=12)
ax.set_ylabel('Total Return (%)', fontsize=12)
ax.set_title('Dual Momentum: Performance vs ROC Lookback Period', fontsize=14)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print best periods
best_top1 = max(results['top1'], key=lambda x: x[1])
best_top2 = max(results['top2'], key=lambda x: x[1])
print(f"\nBest ROC lookback: Top1 = {best_top1[0]} bars ({best_top1[1]*100:.2f}%), Top2 = {best_top2[0]} bars ({best_top2[1]*100:.2f}%)")

Running ROC sweep over 248 periods...
Lookback    1: top1= -94.20%, top2= -99.80%
Lookback    2: top1= -95.37%, top2= -99.88%
Lookback    3: top1= -91.51%, top2= -99.57%
Lookback    4: top1= -76.73%, top2= -97.69%
Lookback    5: top1= -36.57%, top2= -98.51%
Lookback    6: top1= -11.98%, top2= -92.77%


## Step 3: Analyze Current Allocation

Show current allocation and recent history:

In [7]:
current_alloc = get_current_allocation(df_allocations)
print("Current Allocation:")
for etf, alloc in current_alloc.items():
    print(f"  {etf}: {alloc:.2%}")

print_allocation_summary(df_allocations, p_n_last=10)

Current Allocation:
  QQQ: 0.00%
  SPY: 0.00%
  TLT: 0.00%
  GLD: 50.00%
  VWO: 50.00%

Recent Allocations (last 10 periods)
  48097: GLD:0.50, VWO:0.50 | top: GLD | +mom: 3
  48098: GLD:0.50, VWO:0.50 | top: GLD | +mom: 3
  48099: GLD:0.50, VWO:0.50 | top: GLD | +mom: 3
  48100: GLD:0.50, VWO:0.50 | top: GLD | +mom: 3
  48101: GLD:0.50, VWO:0.50 | top: GLD | +mom: 3
  48102: GLD:0.50, VWO:0.50 | top: GLD | +mom: 3
  48103: GLD:0.50, VWO:0.50 | top: GLD | +mom: 3
  48104: GLD:0.50, VWO:0.50 | top: GLD | +mom: 3
  48105: GLD:0.50, VWO:0.50 | top: GLD | +mom: 3
  48106: GLD:0.50, VWO:0.50 | top: GLD | +mom: 3


## Step 4: Run Backtest

Generate orders and compute performance metrics:

In [8]:
orders_df, orders_list = generate_orders_from_allocations(
    p_df=df_allocations,
    p_rebalance_threshold=BACKTEST_PARAMS['rebalance_threshold'],
)

print(f"Generated {len(orders_df)} orders")
print(f"\nOrders sample:")
print(orders_df.head(10))

Generated 5340 orders

Orders sample:
   timestamp  etf  direction      size       price  allocation
0          0  TLT        1.0  0.009370  106.719162         1.0
1        460  QQQ        1.0  0.012792   39.086220         0.5
2        460  TLT       -1.0  0.004394  113.793465         0.5
3        467  QQQ       -1.0  0.012919   38.701885         0.0
4        467  TLT        1.0  0.004403  113.562859         1.0
5        469  QQQ        1.0  0.012881   38.816135         0.5
6        469  TLT       -1.0  0.004409  113.405144         0.5
7        481  QQQ       -1.0  0.012715   39.325089         0.0
8        481  VWO        1.0  0.005017   99.655098         0.5
9        482  QQQ        1.0  0.012701   39.366665         0.5


In [9]:
metrics, df_backtest = compute_strategy_performance(
    p_df=df_allocations,
    p_orders_df=orders_df,
)

print(f"Backtest result shape: {df_backtest.shape}")
print(f"\nNew columns: {[c for c in df_backtest.columns if c not in df_allocations.columns]}")

Backtest result shape: (48107, 1851)

New columns: ['pnl_per_bar', 'cum_pnl', 'portfolio_value']


## Step 5: Performance Metrics

Display strategy performance metrics:

In [10]:
print_metrics(metrics)

metrics_df = pd.DataFrame([metrics]).T
metrics_df.columns = ['Value']
print("\nMetrics DataFrame:")
print(metrics_df)


STRATEGY PERFORMANCE METRICS

                          Returns                           
------------------------------------------------------------
  Total PnL:              12.5957 (1259.57%)
  Final Portfolio Value:  4.1370
  PnL per Bar:            0.000067
  PnL per Month:          0.1885

                        Risk Metrics                        
------------------------------------------------------------
  Max Drawdown:           -0.2513 (-25.13%)
  Sharpe Ratio:           4.4726
  Calmar Ratio:           0.7778

                      Trade Statistics                      
------------------------------------------------------------
  Number of Trades:       5340
  Win Rate:               45.73%
  Avg Win:                0.010771
  Avg Loss:               -0.006832
  Avg Win/Loss Ratio:     1.5765
  Profit Factor:          1.3284

                       Bar Statistics                       
------------------------------------------------------------
  Total Bars:        

## Step 6: Period Returns

Compute monthly and yearly returns:

In [11]:
try:
    monthly_returns, yearly_returns = compute_period_returns(df_backtest)
except ValueError:
    df_backtest.index = pd.date_range(start="2020-01-01", periods=len(df_backtest), freq="h")
    monthly_returns, yearly_returns = compute_period_returns(df_backtest)

print(f"Monthly returns: {monthly_returns.shape}")
print(f"Yearly returns: {yearly_returns.shape}")

print("\nMonthly Returns (last 12 months):")
print(monthly_returns.tail(12))

print("\nYearly Returns:")
print(yearly_returns)

Monthly returns: (1, 6)
Yearly returns: (1, 6)

Monthly Returns (last 12 months):
     month     return  n_bars  positive_bars  negative_bars  win_rate
0  2000-01  21.700735   48107          32136          14902  0.668011

Yearly Returns:
   year     return  n_bars  positive_bars  negative_bars  win_rate
0  2000  21.700735   48107          32136          14902  0.668011


## Step 7: Generate Visualizations

Create and save all charts:

In [12]:
print("Generating visualizations...\n")

plot_normalized_prices(df, OUTPUT_DIR / "normalized_prices.png")
plot_allocation_history(df_backtest, OUTPUT_DIR / "allocation_history.png")
plot_portfolio_value(df_backtest, OUTPUT_DIR / "portfolio_value.png")
plot_cumulative_pnl(df_backtest, OUTPUT_DIR / "cumulative_pnl.png")
plot_pnl_histogram(df_backtest, OUTPUT_DIR / "pnl_histogram.png")
plot_monthly_returns_chart(monthly_returns, OUTPUT_DIR / "monthly_returns.png")
create_allocation_gif(df_allocations, OUTPUT_DIR / "allocation_evolution.gif", p_fps=10)

print(f"\nAll charts saved to: {OUTPUT_DIR}")

Generating visualizations...

Saved: /data/work/dual_momentum/normalized_prices.png
Saved allocation history: /data/work/dual_momentum/allocation_history.png
Saved portfolio value chart: /data/work/dual_momentum/portfolio_value.png
Saved cumulative PnL chart: /data/work/dual_momentum/cumulative_pnl.png
Saved PnL histogram: /data/work/dual_momentum/pnl_histogram.png
Saved monthly returns chart: /data/work/dual_momentum/monthly_returns.png
Saved: /data/work/dual_momentum/allocation_evolution.gif

All charts saved to: /data/work/dual_momentum


## Step 8: Save Diagnostics

Save all backtest results to files:

In [13]:
save_backtest_diagnostics(
    p_result_df=df_backtest,
    p_orders_df=orders_df,
    p_monthly_returns=monthly_returns,
    p_yearly_returns=yearly_returns,
    p_metrics=metrics,
    p_output_dir=OUTPUT_DIR,
)

print(f"\nDiagnostics saved to: {OUTPUT_DIR}")

Saved orders: /data/work/dual_momentum/orders.csv
Saved backtest result: /data/work/dual_momentum/backtest_result.parquet
Saved monthly returns: /data/work/dual_momentum/monthly_returns.csv
Saved yearly returns: /data/work/dual_momentum/yearly_returns.csv
Saved metrics: /data/work/dual_momentum/metrics.txt

Diagnostics saved to: /data/work/dual_momentum


## Summary

### Exposed Variables

**DataFrames:**
- `df` - Raw input data
- `df_allocations` - Data with allocation columns (output of compute_dual_momentum)
- `df_backtest` - Full backtest results with PnL columns
- `orders_df` - Generated orders
- `monthly_returns` - Monthly returns summary
- `yearly_returns` - Yearly returns summary

**Dicts:**
- `current_alloc` - Current portfolio allocation
- `metrics` - Performance metrics dictionary
- `STRATEGY_PARAMS` - Strategy configuration
- `BACKTEST_PARAMS` - Backtest configuration

### Output Files

Charts:
- `normalized_prices.png` - Normalized ETF prices
- `allocation_history.png` - Stacked allocation history
- `portfolio_value.png` - Portfolio value with drawdown
- `cumulative_pnl.png` - Cumulative PnL
- `pnl_histogram.png` - PnL distribution
- `monthly_returns.png` - Monthly returns bar chart
- `allocation_evolution.gif` - Animated allocation GIF

Data files:
- `orders.csv` - All generated orders
- `backtest_result.parquet` - Full backtest results
- `monthly_returns.csv` - Monthly returns data
- `yearly_returns.csv` - Yearly returns data
- `metrics.txt` - Performance metrics

In [34]:
print("=== DIAGNOSTIC ===")
print(f"Allocations sum per row (should be ~1.0):")
alloc_cols = [f"A_{etf}_alloc" for etf in ["QQQ", "SPY", "TLT", "GLD", "VWO"]]
print(df_allocations[alloc_cols].sum(axis=1).describe())
print(f"\nClose prices sample:")
close_cols = [f"{etf}_S_close_f32" for etf in ["QQQ", "SPY", "TLT", "GLD", "VWO"]]
print(df[close_cols].head(10))
print(f"\nPct change sample for QQQ:")
print(df["QQQ_S_close_f32"].pct_change().describe())

=== DIAGNOSTIC ===
Allocations sum per row (should be ~1.0):
count    48107.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
dtype: float64

Close prices sample:
   QQQ_S_close_f32  SPY_S_close_f32  TLT_S_close_f32  GLD_S_close_f32  \
0        39.023914        86.905136       106.719162       102.060188   
1        38.722614        86.568451       106.442993       101.967590   
2        38.681034        86.539825       106.298836       102.337959   
3        38.836868        86.661530       106.586922       102.176620   
4        38.992699        86.962364       106.742996       102.291664   
5        38.930389        86.840668       107.115189       102.337959   
6        38.982334        86.804817       106.875237       102.222221   
7        39.210838        87.062683       106.671036       102.685188   
8        38.816135        86.554192       106.298836       102.893517   
9        38.743465        86.575684 

In [26]:
df.columns

Index(['QQQ_S_open_f32', 'QQQ_S_high_f32', 'QQQ_S_low_f32', 'QQQ_S_close_f32',
       'QQQ_S_volume_f64', 'QQQ_S_open_time_i', 'QQQ_S_close_time_i',
       'QQQ_minute_diff', 'QQQ_gap_flag', 'QQQ_valid_row',
       ...
       'VWO_F_diff_rel_ema_2_F_mid_f32_2_f16',
       'VWO_F_diff_rel_ema_2_F_mid_f32_15_f16',
       'VWO_F_diff_rel_ema_2_F_mid_f32_60_f16',
       'VWO_F_diff_rel_ema_2_F_mid_f32_240_f16',
       'VWO_F_diff_rel_ema_2_F_mid_f32_500_f16',
       'VWO_F_diff_rel_ema_2_F_mid_f32_1440_f16', 'VWO_F_acc_dist_index_f64',
       'VWO_F_roc_14_F_mid_f32_f16', 'VWO_F_roc_60_F_mid_f32_f16',
       'VWO_F_roc_240_F_mid_f32_f16'],
      dtype='str', length=611)

In [27]:
df_allocations.columns

Index(['QQQ_S_open_f32', 'QQQ_S_high_f32', 'QQQ_S_low_f32', 'QQQ_S_close_f32',
       'QQQ_S_volume_f64', 'QQQ_S_open_time_i', 'QQQ_S_close_time_i',
       'QQQ_minute_diff', 'QQQ_gap_flag', 'QQQ_valid_row',
       ...
       'A_TLT_alloc', 'A_GLD_alloc', 'A_VWO_alloc', 'A_top_etf',
       'A_n_positive_momentum', 'A_QQQ_roc_rank', 'A_SPY_roc_rank',
       'A_TLT_roc_rank', 'A_GLD_roc_rank', 'A_VWO_roc_rank'],
      dtype='str', length=623)